In [2]:
import pandas as pd

df_rank = pd.read_csv('../catboost-classifier/datasets/rank_dataset_TRAIN_CATBOOST.csv')
df_mapping = pd.read_csv('./exercises_base_FINAL_CLEANED.csv')
df_profiles = pd.read_csv('./powerlifting_profiles_clean.csv')

df_stage_1 = df_rank.merge(
    df_mapping[['exercise_id', 'Base_Lift', 'Exercise_Name']], 
    left_on='candidate_id', 
    right_on='exercise_id', 
    how='inner' # Оставляем только то, что разметили и что есть в трейне
)

final_master_df = df_stage_1.join(df_profiles, how='inner')

final_master_df.to_csv('MASTER_CATBOOST_DATASET.csv', index=False)

print(f"Сборка завершена!")
print(f"Финальное количество строк: {len(final_master_df)}")
display(final_master_df.head())

Сборка завершена!
Финальное количество строк: 45239


,bert_score,candidate_id,is_same_group,target,eq_clean,level_idx,goal_clean,exercise_id,Base_Lift,Exercise_Name,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg
0,0.001278,1546,0,0,Machine,3,Powerbuilding,1546,Best3BenchKg,Low Cable Fly,F,33.0,58.30,80.0,60.0,107.5
1,0.001276,1440,0,0,Machine,0,Bodybuilding,1440,Best3DeadliftKg,Lat Pullover,F,43.0,73.10,105.0,67.5,110.0
2,0.001778,2733,0,0,Machine,0,Athletics,2733,Best3DeadliftKg,Super ROM Overhead Cable Row,M,15.5,67.40,100.0,62.5,105.0
3,0.001300,2731,0,0,Dumbbell,0,Athletics,2731,Best3DeadliftKg,Super ROM Lateral Dumbbell Raise,M,35.0,66.65,137.5,122.5,170.0
4,0.001246,1075,0,0,Barbell,1,Athletics,1075,Best3BenchKg,Half Kneeling Landmine Press,M,26.5,72.45,90.0,50.0,125.0


In [3]:
features_with_id = [
    'exercise_id',
    'Sex', 'Age', 'BodyweightKg', 
    'eq_clean', 'level_idx', 'goal_clean', 
    'Base_Lift', 
    'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg'
]

df_final_reg = final_master_df[features_with_id].copy()

df_final_reg['exercise_id'] = df_final_reg['exercise_id'].astype(int)

print(f"Теперь всё на месте. Размер: {df_final_reg.shape}")
display(df_final_reg.head(20))

Теперь всё на месте. Размер: (45239, 11)


,exercise_id,Sex,Age,BodyweightKg,eq_clean,level_idx,goal_clean,Base_Lift,Best3SquatKg,Best3BenchKg,Best3DeadliftKg
0,1546,F,33.0,58.30,Machine,3,Powerbuilding,Best3BenchKg,80.0,60.0,107.5
1,1440,F,43.0,73.10,Machine,0,Bodybuilding,Best3DeadliftKg,105.0,67.5,110.0
2,2733,M,15.5,67.40,Machine,0,Athletics,Best3DeadliftKg,100.0,62.5,105.0
3,2731,M,35.0,66.65,Dumbbell,0,Athletics,Best3DeadliftKg,137.5,122.5,170.0
4,1075,M,26.5,72.45,Barbell,1,Athletics,Best3BenchKg,90.0,50.0,125.0
5,1134,M,15.5,78.80,Dumbbell,2,Powerbuilding,Best3DeadliftKg,100.0,60.0,115.0
6,2577,M,57.5,79.65,Dumbbell,2,Powerlifting,Best3DeadliftKg,180.0,100.0,55.0
7,185,F,26.0,96.50,Bodyweight,2,Muscle & Sculpting,Cardio_Core,100.0,47.5,140.0
8,2703,M,31.5,102.55,Barbell,2,Powerlifting,Best3DeadliftKg,232.5,160.0,260.0
9,1472,F,35.0,53.60,Dumbbell,0,Muscle & Sculpting,Best3DeadliftKg,80.0,50.0,92.5


In [4]:
# Нам нужно вытащить Exercise_Name из твоего маппинга по exercise_id
# Используем merge, чтобы точно сопоставить названия с ID
df_final_reg = df_final_reg.merge(
    df_mapping[['exercise_id', 'Exercise_Name']], 
    on='exercise_id', 
    how='left'
)

# Переставим колонки, чтобы название было сразу после ID для удобства
cols = ['exercise_id', 'Exercise_Name'] + [c for c in df_final_reg.columns if c not in ['exercise_id', 'Exercise_Name']]
df_final_reg = df_final_reg[cols]

print(f"Теперь таблица 'говорящая'. Размер: {df_final_reg.shape}")
display(df_final_reg.head())

Теперь таблица 'говорящая'. Размер: (45239, 12)


,exercise_id,Exercise_Name,Sex,Age,BodyweightKg,eq_clean,level_idx,goal_clean,Base_Lift,Best3SquatKg,Best3BenchKg,Best3DeadliftKg
0,1546,Low Cable Fly,F,33.0,58.30,Machine,3,Powerbuilding,Best3BenchKg,80.0,60.0,107.5
1,1440,Lat Pullover,F,43.0,73.10,Machine,0,Bodybuilding,Best3DeadliftKg,105.0,67.5,110.0
2,2733,Super ROM Overhead Cable Row,M,15.5,67.40,Machine,0,Athletics,Best3DeadliftKg,100.0,62.5,105.0
3,2731,Super ROM Lateral Dumbbell Raise,M,35.0,66.65,Dumbbell,0,Athletics,Best3DeadliftKg,137.5,122.5,170.0
4,1075,Half Kneeling Landmine Press,M,26.5,72.45,Barbell,1,Athletics,Best3BenchKg,90.0,50.0,125.0


In [5]:
import pandas as pd
import numpy as np

# # 1. ФУНКЦИЯ ИНТЕГРИРОВАННОЙ ТРЕНЕРСКОЙ ЛОГИКИ
# def calculate_pro_stats_integrated(row):
#     # --- БАЗОВАЯ ЛОГИКА ЦЕЛЕЙ (Повторы и Интенсивность) ---
#     goal_logic = {
#         'Powerlifting': {'reps': 3, 'int': 0.85},
#         'Bodybuilding': {'reps': 10, 'int': 0.70},
#         'Athletics': {'reps': 12, 'int': 0.60},
#         'Powerbuilding': {'reps': 8, 'int': 0.75},
#         'Muscle & Sculpting': {'reps': 15, 'int': 0.60},
#         'Fitness': {'reps': 12, 'int': 0.65},
#         'Bodyweight Fitness': {'reps': 12, 'int': 0.60}
#     }
#     params = goal_logic.get(row['goal_clean'], {'reps': 10, 'int': 0.70})
    
#     # --- ПОПРАВКИ АТЛЕТА ---
#     level_mult = 0.85 + (row['level_idx'] * 0.05)  # Новичок (0.85) -> Профи (1.0)
#     age_mult = 1.0 - (max(0, row['Age'] - 40) * 0.005) # -0.5% за каждый год после 40
#     sex_rep_adj = 2 if row['Sex'] == 'F' else 0 # Женщинам +2 повтора к базе
#     total_reps = params['reps'] + sex_rep_adj
    
#     # --- БИОМЕХАНИКА УПРАЖНЕНИЯ ---
#     base_ratios = {
#         'Best3SquatKg': 0.75, 
#         'Best3BenchKg': 0.60, 
#         'Best3DeadliftKg': 0.70, 
#         'Cardio_Core': 0.15
#     }
#     ex_ratio = base_ratios.get(row['Base_Lift'], 0.5)
    
#     # --- РАСЧЕТ 1RM ДЛЯ УПРАЖНЕНИЯ ---
#     base_max = row[row['Base_Lift']] if row['Base_Lift'] in row and not pd.isna(row[row['Base_Lift']]) else 0
#     if base_max == 0: return 0.0, 0
    
#     # Коэффициент оборудования
#     eq_mult = {'Barbell': 1.0, 'Dumbbell': 0.85, 'Machine': 1.15, 'Bodyweight': 1.0}.get(row['eq_clean'], 1.0)
    
#     # Абсолютный потенциал в этом упражнении (1RM эквивалент)
#     est_1rm_capacity = base_max * ex_ratio * eq_mult
    
#     # Целевая абсолютная нагрузка (Total Load) с учетом цели, уровня и возраста
#     # Используем формулу Эйли для пересчета 1RM в рабочий вес
#     target_absolute_load = (est_1rm_capacity * params['int'] * level_mult * age_mult) / (1 + total_reps / 30)
    
#     # --- ЛОГИКА ВЕСА ТЕЛА (BODYWEIGHT) ---
#     if row['eq_clean'] == 'Bodyweight':
#         # Сколько % веса тела участвует в движении
#         bw_map = {
#             'pull': 0.95, 'chin': 0.95, 'dip': 0.90, 'push': 0.65, 
#             'squat': 0.70, 'lunge': 0.50, 'leg raise': 0.30, 'plank': 0.50,
#             'skips': 0.60, 'pogo': 0.70, 'march': 0.50, 'crunch': 0.40, 'hip thrust': 0.80
#         }
#         k = 0.8 # дефолт
#         name_lower = str(row['Exercise_Name']).lower()
#         for key, val in bw_map.items():
#             if key in name_lower:
#                 k = val
#                 break
        
#         # Предсказанный доп. вес = Целевая нагрузка - Эффективный вес тела
#         # Если < 0, значит нужна помощь (резина/гравитрон)
#         predicted_weight = target_absolute_load - (row['BodyweightKg'] * k)
#     else:
#         # Для снарядов - это чистый вес на штанге/тренажере
#         predicted_weight = target_absolute_load
        
#     return round(predicted_weight, 1), int(total_reps)

import pandas as pd
import numpy as np

def calculate_pro_stats_integrated(row):
    # --- 1. БАЗОВАЯ ЛОГИКА ЦЕЛЕЙ ---
    goal_logic = {
        'Powerlifting': {'reps': 3, 'int': 0.85},
        'Bodybuilding': {'reps': 10, 'int': 0.70},
        'Athletics': {'reps': 12, 'int': 0.60},
        'Powerbuilding': {'reps': 8, 'int': 0.75},
        'Muscle & Sculpting': {'reps': 15, 'int': 0.60},
        'Fitness': {'reps': 12, 'int': 0.65},
        'Bodyweight Fitness': {'reps': 12, 'int': 0.60}
    }
    params = goal_logic.get(row['goal_clean'], {'reps': 10, 'int': 0.70})
    
    # --- 2. ПОПРАВКИ АТЛЕТА ---
    level_mult = 0.85 + (row['level_idx'] * 0.05) 
    age_mult = 1.0 - (max(0, row['Age'] - 40) * 0.005)
    sex_rep_adj = 2 if row['Sex'] == 'F' else 0
    total_reps = params['reps'] + sex_rep_adj
    
    # --- 3. БИОМЕХАНИКА (ИСПРАВЛЕНО) ---
    base_ratios = {
        'Best3SquatKg': 0.80, 
        'Best3BenchKg': 0.75, 
        'Best3DeadliftKg': 0.85, 
        'Cardio_Core': 0.15
    }
    
    # ПРОВЕРКА: Если название упражнения содержит базу, не режем коэффициент сильно
    ex_name_lower = str(row['Exercise_Name']).lower()
    is_main_lift = any(x in ex_name_lower for x in ['bench press', 'squat', 'deadlift']) and 'dumbbell' not in ex_name_lower
    
    if is_main_lift:
        ex_ratio = 1.0 # База к базе = 100%
    else:
        ex_ratio = base_ratios.get(row['Base_Lift'], 0.5)
    
    # --- 4. РАСЧЕТ 1RM ---
    base_max = row[row['Base_Lift']] if row['Base_Lift'] in row and not pd.isna(row[row['Base_Lift']]) else 0
    if base_max == 0: return 0.0, int(total_reps)
    
    eq_mult = {'Barbell': 1.0, 'Dumbbell': 0.85, 'Machine': 1.10, 'Bodyweight': 1.0}.get(row['eq_clean'], 1.0)
    
    # Эквивалентный 1RM для конкретного упражнения
    est_1rm_capacity = base_max * ex_ratio * eq_mult
    
    # --- 5. РАБОЧИЙ ВЕС (ИСПРАВЛЕНО) ---
    # УБРАЛИ деление на (1 + reps/30), так как params['int'] уже учитывает работу на повторы
    predicted_weight = est_1rm_capacity * params['int'] * level_mult * age_mult
    
    # --- 6. ВЕС ТЕЛА (BODYWEIGHT) ---
    if row['eq_clean'] == 'Bodyweight':
        bw_map = {'pull': 0.95, 'push': 0.65, 'squat': 0.70, 'crunch': 0.40, 'hip thrust': 0.80}
        k = 0.8
        for key, val in bw_map.items():
            if key in ex_name_lower:
                k = val
                break
        predicted_weight = predicted_weight - (row['BodyweightKg'] * k)
    
    return round(max(0, predicted_weight), 1), int(total_reps)
# 2. ПОДГОТОВКА И ВЫВОД ТЕСТОВОЙ ВЫБОРКИ
# Список колонок, которые мы хотим видеть для анализа
analysis_cols = [
    'Exercise_Name', 'Sex', 'Age', 'BodyweightKg', 
    'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', # Силовые в центре внимания
    'goal_clean', 'level_idx', 'Base_Lift', 'eq_clean',
    'predicted_weight', 'target_reps'
]

# Применяем расчет к 100 рандомным строкам
df_sample = df_final_reg.sample(n=100, random_state=42).copy()
res = df_sample.apply(lambda x: calculate_pro_stats_integrated(x), axis=1)
df_sample['predicted_weight'] = [x[0] for x in res]
df_sample['target_reps'] = [x[1] for x in res]

# Выводим результат
display(df_sample[analysis_cols].head(50))
# 2. ПОДГОТОВКА И ВЫВОД ТЕСТОВОЙ ВЫБОРКИ
# Список колонок, которые мы хотим видеть для анализа
analysis_cols = [
    'Exercise_Name', 'Sex', 'Age', 'BodyweightKg', 
    'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', # Силовые в центре внимания
    'goal_clean', 'level_idx', 'Base_Lift', 'eq_clean',
    'predicted_weight', 'target_reps'
]

# Применяем расчет к 100 рандомным строкам
df_sample = df_final_reg.sample(n=100, random_state=42).copy()
res = df_sample.apply(lambda x: calculate_pro_stats_integrated(x), axis=1)
df_sample['predicted_weight'] = [x[0] for x in res]
df_sample['target_reps'] = [x[1] for x in res]

# Выводим результат
display(df_sample[analysis_cols].head(50))

,Exercise_Name,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,goal_clean,level_idx,Base_Lift,eq_clean,predicted_weight,target_reps
36660,Lying Overhead Tricep Extension (barbell),M,28.5,114.45,170.0,125.0,175.0,Powerbuilding,0,Best3BenchKg,Machine,65.7,8
24321,Speed Bench Press,M,39.5,103.45,300.0,175.0,285.0,Powerlifting,0,Cardio_Core,Barbell,0.0,3
3460,Incline Dumbbell Row,M,11.0,59.87,97.5,52.5,115.0,Powerlifting,3,Best3DeadliftKg,Dumbbell,70.6,3
23965,Lying Leg Raise,M,50.0,92.40,150.0,107.5,177.5,Bodybuilding,2,Best3BenchKg,Bodyweight,0.0,10
43126,Straight Leg Deadlift,M,58.0,53.50,95.0,75.0,137.5,Powerlifting,2,Best3DeadliftKg,Barbell,101.0,3
26600,Unilateral Preacher Curl,F,22.5,82.80,145.0,90.0,175.0,Muscle & Sculpting,0,Best3DeadliftKg,Dumbbell,64.5,17
25872,Reverse Grip Pushdown,F,40.0,74.60,147.5,70.0,157.5,Powerbuilding,0,Best3BenchKg,Machine,36.8,10
18619,Calf Press (Leg Press Machine),F,46.5,61.00,85.0,57.5,135.0,Powerlifting,2,Best3SquatKg,Machine,58.4,5
13325,Ring Lat Pull,F,21.5,73.60,125.0,70.0,125.0,Powerbuilding,0,Best3DeadliftKg,Bodyweight,0.0,10
38637,Clamshells With Band,M,17.0,80.40,210.0,127.5,230.0,Muscle & Sculpting,0,Cardio_Core,Bodyweight,0.0,15


,Exercise_Name,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,goal_clean,level_idx,Base_Lift,eq_clean,predicted_weight,target_reps
36660,Lying Overhead Tricep Extension (barbell),M,28.5,114.45,170.0,125.0,175.0,Powerbuilding,0,Best3BenchKg,Machine,65.7,8
24321,Speed Bench Press,M,39.5,103.45,300.0,175.0,285.0,Powerlifting,0,Cardio_Core,Barbell,0.0,3
3460,Incline Dumbbell Row,M,11.0,59.87,97.5,52.5,115.0,Powerlifting,3,Best3DeadliftKg,Dumbbell,70.6,3
23965,Lying Leg Raise,M,50.0,92.40,150.0,107.5,177.5,Bodybuilding,2,Best3BenchKg,Bodyweight,0.0,10
43126,Straight Leg Deadlift,M,58.0,53.50,95.0,75.0,137.5,Powerlifting,2,Best3DeadliftKg,Barbell,101.0,3
26600,Unilateral Preacher Curl,F,22.5,82.80,145.0,90.0,175.0,Muscle & Sculpting,0,Best3DeadliftKg,Dumbbell,64.5,17
25872,Reverse Grip Pushdown,F,40.0,74.60,147.5,70.0,157.5,Powerbuilding,0,Best3BenchKg,Machine,36.8,10
18619,Calf Press (Leg Press Machine),F,46.5,61.00,85.0,57.5,135.0,Powerlifting,2,Best3SquatKg,Machine,58.4,5
13325,Ring Lat Pull,F,21.5,73.60,125.0,70.0,125.0,Powerbuilding,0,Best3DeadliftKg,Bodyweight,0.0,10
38637,Clamshells With Band,M,17.0,80.40,210.0,127.5,230.0,Muscle & Sculpting,0,Cardio_Core,Bodyweight,0.0,15


In [6]:
import os

# Применяем расчет ко ВСЕМУ датасету
print(f"🔄 Обработка {len(df_final_reg)} строк. Пожалуйста, подождите...")

# Применяем функцию и разделяем результат на две колонки
results_all = df_final_reg.apply(lambda x: calculate_pro_stats_integrated(x), axis=1)
df_final_reg['predicted_weight'] = [x[0] for x in results_all]
df_final_reg['target_reps'] = [x[1] for x in results_all]

# Формируем имя файла
output_filename = "./datasets/gym_dataset_with_targets_FINAL.csv"

# Сохраняем на диск
df_final_reg.to_csv(output_filename, index=False)

print(f"✅ Готово! Датасет сохранен как: {output_filename}")
print(f"📊 Средний предсказанный вес: {df_final_reg['predicted_weight'].mean():.2f} кг")

🔄 Обработка 45239 строк. Пожалуйста, подождите...
✅ Готово! Датасет сохранен как: ./datasets/gym_dataset_with_targets_FINAL.csv
📊 Средний предсказанный вес: 63.55 кг
